In [1]:
import autoslo.utils.paths as pu
import os
from tqdm.auto import tqdm
import sqlglot
import pandas as pd
from autoslo.featurization.iconq_query_featurizer import IconqQueryFeaturizer

## Let's first try to do it solely based on the query texts.

In [2]:
all_table_names = [
    "customer_address",
    "customer_demographics",
    "date_dim",
    "warehouse",
    "ship_mode",
    "time_dim",
    "reason",
    "income_band",
    "item",
    "store",
    "call_center",
    "customer",
    "web_site",
    "store_returns",
    "household_demographics",
    "web_page",
    "promotion",
    "catalog_page",
    "inventory",
    "catalog_returns",
    "web_returns",
    "web_sales",
    "catalog_sales",
    "store_sales",
]

In [3]:
# Find the table names mentioned in each query text
base_path = pu.QUERIES_PATH

by_text = {}

for temp_num in tqdm(range(1, 100)):
    temp_path = os.path.join(base_path, f'query{temp_num:03d}')
    if not os.path.exists(temp_path):

        continue

    by_text[temp_num] = {}

    for q_in_temp_num in range(1, 4):
        query_path = os.path.join(
            base_path, f'query{temp_num:03d}', f'query{temp_num:03d}_{q_in_temp_num:03d}.sql')
        if not os.path.exists(query_path):
            continue

        with open(query_path, 'r') as f:
            sql = f.read()

        parsed = sqlglot.parse_one(sql)  # Validate SQL syntax

        # Get the tables from the parsed SQL
        found_tables = set()
        for table in parsed.find_all(sqlglot.exp.Table):
            table_name = table.name.lower()
            if table_name in all_table_names: # To avoid CTEs or aliases
                found_tables.add(table_name)

        by_text[temp_num][q_in_temp_num] = found_tables


  0%|          | 0/99 [00:00<?, ?it/s]

In [4]:
# Assert that the tables we found actually exist in the known table names
for temp_num, queries in by_text.items():
    for q_in_temp_num, tables in queries.items():
        for table in tables:
            assert table in all_table_names, f"Unknown table {table} in query {temp_num}_{q_in_temp_num}"

In [5]:
# Check that, per template, all queries use the same tables
for temp_num, queries in by_text.items():
    table_sets = [tables for tables in queries.values()]
    first_set = table_sets[0]
    for other_set in table_sets[1:]:
        assert first_set == other_set, f"Different table sets in query template {temp_num}: {first_set} vs {other_set}"

## Now let's try to do it using the plans

In [6]:
runs_df = pu.RunLocator.get_runs_df()
sub_df = runs_df[runs_df['workload_name'].str.contains('tpcds') & runs_df['schema_name'].str.contains('ext') & runs_df['blueprint_name'].str.contains('single')]
sub_df

,run_id,workload_name,num_queries,scale_factor,schema_name,blueprint_name,query_router_name,maxconns,closed_loop
21,1763240500,tpcds_99templates_00pctheavy_120meaninterarrivals,34,1000,ext_tpcds1000,single_4,RFixed(fixed_cluster_name='cluster_4'),1000,False
22,1763240506,tpcds_99templates_00pctheavy_120meaninterarrivals,34,1000,ext_tpcds1000,single_16,RFixed(fixed_cluster_name='cluster_16'),1000,False
23,1763240707,tpcds_99templates_00pctheavy_120meaninterarrivals,34,1000,ext_tpcds1000,single_8,RFixed(fixed_cluster_name='cluster_8'),1000,False
24,1763244677,tpcds_99templates_00pctheavy_60meaninterarrivals,71,1000,ext_tpcds1000,single_4,RFixed(fixed_cluster_name='cluster_4'),1000,False
25,1763244682,tpcds_99templates_00pctheavy_60meaninterarrivals,71,1000,ext_tpcds1000,single_16,RFixed(fixed_cluster_name='cluster_16'),1000,False
...,...,...,...,...,...,...,...,...,...
86,1763961985,tpcds_99templates_25pctheavy_10meaninterarrivals,347,1000,ext_tpcds1000,single_32,RFixed(fixed_cluster_name='cluster_32'),1000,False
87,1763966257,tpcds_99templates_50pctheavy_120meaninterarrivals,25,1000,ext_tpcds1000,single_32,RFixed(fixed_cluster_name='cluster_32'),1000,False
88,1763970399,tpcds_99templates_50pctheavy_60meaninterarrivals,54,1000,ext_tpcds1000,single_32,RFixed(fixed_cluster_name='cluster_32'),1000,False
89,1763974541,tpcds_99templates_50pctheavy_30meaninterarrivals,114,1000,ext_tpcds1000,single_32,RFixed(fixed_cluster_name='cluster_32'),1000,False


In [12]:
run_ids = list(sub_df['run_id'].unique())
featurizer_old = IconqQueryFeaturizer(schema_name='tpcds1000', run_ids=run_ids,m=5, n=25)

Finding top operators...
`from_sys_query_explain` is False; using query plans.


  0%|          | 0/64 [00:00<?, ?it/s]

  0%|          | 0/9020 [00:00<?, ?it/s]

Top 5 operators cover 192760/275712 = 69.91% of all operator occurrences.
Top operators:
  0: XN Seq Scan
  1: XN Hash
  2: XN Hash Join DS_DIST_ALL_NONE
  3: XN Subquery Scan
  4: XN HashAggregate
Finding top tables...
Top 25 tables by size:
  store_sales: 2879987999 rows
  catalog_sales: 1439980416 rows
  inventory: 783000000 rows
  web_sales: 720000376 rows
  store_returns: 287999764 rows
  catalog_returns: 143996756 rows
  web_returns: 71997522 rows
  customer: 12000000 rows
  customer_address: 6000000 rows
  customer_demographics: 1920800 rows
  item: 300000 rows
  time_dim: 86400 rows
  date_dim: 73049 rows
  catalog_page: 30000 rows
  household_demographics: 7200 rows
  web_page: 3000 rows
  promotion: 1500 rows
  store: 1002 rows
  reason: 65 rows
  web_site: 54 rows
  call_center: 42 rows
  income_band: 20 rows
  ship_mode: 20 rows
  warehouse: 20 rows
Featurizing queries...


  0%|          | 0/64 [00:00<?, ?it/s]

In [23]:
featurizer_new = IconqQueryFeaturizer(schema_name='tpcds1000', run_ids=run_ids,m=5, n=25, from_sys_query_explain=True)

Finding top operators...
`from_sys_query_explain` is True; using sys_query_explain.


  0%|          | 0/64 [00:00<?, ?it/s]

Top 5 operators cover 208140/294937 = 70.57% of all operator occurrences.
Top operators:
  0: XN Seq Scan
  1: XN Hash
  2: XN Hash Join DS_DIST_ALL_NONE
  3: XN Subquery Scan
  4: XN HashAggregate
Finding top tables...
Top 25 tables by size:
  store_sales: 2879987999 rows
  catalog_sales: 1439980416 rows
  inventory: 783000000 rows
  web_sales: 720000376 rows
  store_returns: 287999764 rows
  catalog_returns: 143996756 rows
  web_returns: 71997522 rows
  customer: 12000000 rows
  customer_address: 6000000 rows
  customer_demographics: 1920800 rows
  item: 300000 rows
  time_dim: 86400 rows
  date_dim: 73049 rows
  catalog_page: 30000 rows
  household_demographics: 7200 rows
  web_page: 3000 rows
  promotion: 1500 rows
  store: 1002 rows
  reason: 65 rows
  web_site: 54 rows
  call_center: 42 rows
  income_band: 20 rows
  ship_mode: 20 rows
  warehouse: 20 rows
Featurizing queries...


  0%|          | 0/64 [00:00<?, ?it/s]

In [ ]:
from autoslo.workload_execution.trace import Trace
results = []

for _, row in tqdm(sub_df.iterrows(), total=len(sub_df)):
    run_id = row['run_id']
    trace = Trace(run_id)

    plans = trace.query_plans(ignore_caching=True)
    tpcds_temp_and_q_idxs = trace.tpcds_temp_and_q_idxs

    sys_query_explain_df = list(trace._dfs['sys_query_explain'].values())[0]
    was_aborted = trace.was_aborted()
    seq_nums = trace.seq_nums
    error_messages = trace.error_messages()

    for (query_id, tpcds_temp_and_q_idx) in tpcds_temp_and_q_idxs.items():
        sys_query_explain_rows = sys_query_explain_df[sys_query_explain_df['query_id'] == query_id]

        error_message = error_messages[query_id].strip()
        child_queries_to_ignore = set()
        if len(error_message) > 0 and "child_sequence:" in error_message:
            # The digit following "child_sequence"
            child_queries_to_ignore.add(int(error_message.split("child_sequence:")[-1][0]))


        featurization = featurizer.featurize_plan_from_sys_query_explain_rows(
            sys_query_explain_sub_df=sys_query_explain_rows,
            child_queries_to_ignore=child_queries_to_ignore,
        )
        temp_num = Trace.extract_temp(tpcds_temp_and_q_idx)
        q_in_temp_num = Trace.extract_q_idx(tpcds_temp_and_q_idx)

        d = {
            "run_id": run_id,
            "workload_name": row['workload_name'],  
            "blueprint_name": row['blueprint_name'],
            "query_id": query_id,
            "seq_num": seq_nums[query_id],
            "temp_num": temp_num,
            "q_in_temp_num": q_in_temp_num,
            "was_aborted": was_aborted[query_id],
            "featurization": featurization,
            "found_all_tables": True,
            "error_message": error_message,
        }

        for table in by_text[temp_num][q_in_temp_num]:
            if not featurizer.nonzero_feature_for_table(featurization=featurization, table_name=table):
                d["found_all_tables"] = False

        results.append(d)
                    

In [24]:
from autoslo.workload_execution.trace import Trace

results_new = []

for _, row in tqdm(sub_df.iterrows(), total=len(sub_df)):
    run_id = row["run_id"]
    trace = Trace(run_id)

    plans = trace.query_plans(ignore_caching=True)
    tpcds_temp_and_q_idxs = trace.tpcds_temp_and_q_idxs

    sys_query_explain_df = list(trace._dfs["sys_query_explain"].values())[0]
    was_aborted = trace.was_aborted()
    seq_nums = trace.seq_nums
    error_messages = trace.error_messages()

    sys_query_explain_rows_per_query = trace.sys_query_explain_rows_per_query()

    for (
        query_id,
        sys_query_explain_rows,
    ) in sys_query_explain_rows_per_query.items():
        
        
        tpcds_temp_and_q_idx = tpcds_temp_and_q_idxs[query_id]
        error_message = error_messages[query_id].strip()

        featurization = (
            featurizer_new.featurize_plan_from_sys_query_explain_rows(
                sys_query_explain_sub_df=sys_query_explain_rows,
            )
        )
        temp_num = Trace.extract_temp(tpcds_temp_and_q_idx)
        q_in_temp_num = Trace.extract_q_idx(tpcds_temp_and_q_idx)

        d = {
            "run_id": run_id,
            "workload_name": row["workload_name"],
            "blueprint_name": row["blueprint_name"],
            "query_id": query_id,
            "seq_num": seq_nums[query_id],
            "temp_num": temp_num,
            "q_in_temp_num": q_in_temp_num,
            "was_aborted": was_aborted[query_id],
            "featurization": featurization,
            "found_all_tables": True,
            "error_message": error_message,
        }

        for table in by_text[temp_num][q_in_temp_num]:
            if not featurizer_new.nonzero_feature_for_table(
                featurization=featurization, table_name=table
            ):
                d["found_all_tables"] = False

        results_new.append(d)

results_df_new = pd.DataFrame(results_new)

  0%|          | 0/64 [00:00<?, ?it/s]

In [25]:
results_df_new['found_all_tables'].value_counts()

found_all_tables
True     8935
False      85
Name: count, dtype: int64

In [26]:
results_df_new['has_error_message'] = results_df_new['error_message'].apply(lambda x: x is not None and x != '')
results_df_new[['found_all_tables', 'was_aborted']].value_counts()

found_all_tables  was_aborted
True              False          8796
                  True            139
False             True             85
Name: count, dtype: int64

In [168]:
# After grouping by temp_num, q_in_temp_num, blueprint_name, are all the featurizations the same
from math import isclose, trunc
for group_name, group in results_df[~results_df['was_aborted']].groupby(['temp_num', 'q_in_temp_num']):
    first_featurization = group['featurization'].iloc[0]
    for other_featurization in group['featurization'].iloc[1:]:
        all_elements_close = all(round(a, 1) == round(b, 1) for a, b in zip(first_featurization, other_featurization))
        if not all_elements_close:
            print(f"Different featurizations in group {group_name}")
            print(first_featurization)
            print(other_featurization)
            break

Different featurizations in group (np.int64(44), np.int64(1))
[6.0, 19.31621220549331, 3.0, 9.353574540928642, 0.0, 0.0, 4.0, 10.695574681177426, 4.0, 10.004237456088351, 19.239859866421817, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 16.705882315860993, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
[6.0, 19.31621220549331, 3.0, 9.30819277305047, 0.0, 0.0, 4.0, 10.695574681177426, 4.0, 10.004237456088351, 19.239859866421817, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 16.705882315860993, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
Different featurizations in group (np.int64(44), np.int64(3))
[6.0, 19.314838416782482, 3.0, 9.30819277305047, 0.0, 0.0, 4.0, 10.695574681177426, 4.0, 10.004237456088351, 19.238376996610498, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 16.705882315860993, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
[6.0, 19.314838416782482, 3.0, 9.353574540928642, 0.0, 0.0, 4.0, 10.695574681177426, 

In [143]:
results_df[(results_df['temp_num'] == 1) & (results_df['q_in_temp_num'] == 1)] #& (results_df['blueprint_name'] == 'single_4')]

,run_id,workload_name,blueprint_name,query_id,seq_num,temp_num,q_in_temp_num,was_aborted,featurization,found_all_tables
405,1763248884,tpcds_99templates_00pctheavy_30meaninterarrivals,single_16,cluster_16_10767437#b24232eb-8c14-452e-b77f-571711cd39d3,90,1,1,False,"[8.0, 23.262456591536385, 6.0, 15.590970576602743, 3.0, 21.122383723998116, 3.0, 18.51808262553164, 3.0, 20.59735201283514, 0.0, 0.0, 0.0, 0.0, 23.123069142836815, 0.0, 0.0, 19.478471038100256, 0.0, 0.0, 0.0, 0.0, 7.943072720829069, 0.0, 0.0, 0.0, 0.0, 5.676753836514856, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]",True
532,1763249004,tpcds_99templates_00pctheavy_30meaninterarrivals,single_4,cluster_4_4436892#fafa3bce-e75d-484b-81e9-97b059599c54,90,1,1,False,"[8.0, 23.262456591536385, 6.0, 15.590845208673118, 3.0, 21.122383723998116, 3.0, 18.51808262553164, 3.0, 20.59735201283514, 0.0, 0.0, 0.0, 0.0, 23.123069142836815, 0.0, 0.0, 19.478471038100256, 0.0, 0.0, 0.0, 0.0, 7.943072720829069, 0.0, 0.0, 0.0, 0.0, 5.676753836514856, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]",True
659,1763249132,tpcds_99templates_00pctheavy_30meaninterarrivals,single_8,cluster_8_4474202#8215d3c6-c6f0-47b9-9659-00f5ac9dcdaf,90,1,1,False,"[8.0, 23.262456591536385, 6.0, 15.590814711394659, 3.0, 21.122383723998116, 3.0, 18.51808262553164, 3.0, 20.59735201283514, 0.0, 0.0, 0.0, 0.0, 23.123069142836815, 0.0, 0.0, 19.478471038100256, 0.0, 0.0, 0.0, 0.0, 7.943072720829069, 0.0, 0.0, 0.0, 0.0, 5.676753836514856, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]",True
786,1763253072,tpcds_99templates_00pctheavy_10meaninterarrivals,single_16,cluster_16_10969733#8b49bae4-b9b3-466d-82f6-b8db25a8e000,90,1,1,False,"[8.0, 23.262456591536385, 6.0, 15.590663566710614, 3.0, 21.122383723998116, 3.0, 18.51808262553164, 3.0, 20.59735201283514, 0.0, 0.0, 0.0, 0.0, 23.123069142836815, 0.0, 0.0, 19.478471038100256, 0.0, 0.0, 0.0, 0.0, 7.943072720829069, 0.0, 0.0, 0.0, 0.0, 5.676753836514856, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]",True
833,1763253072,tpcds_99templates_00pctheavy_10meaninterarrivals,single_16,cluster_16_10970210#8b49bae4-b9b3-466d-82f6-b8db25a8e000,137,1,1,False,"[8.0, 23.262456591536385, 6.0, 15.590754391816091, 3.0, 21.122383723998116, 3.0, 18.51808262553164, 3.0, 20.59735201283514, 0.0, 0.0, 0.0, 0.0, 23.123069142836815, 0.0, 0.0, 19.478471038100256, 0.0, 0.0, 0.0, 0.0, 7.943072720829069, 0.0, 0.0, 0.0, 0.0, 5.676753836514856, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]",True
929,1763253072,tpcds_99templates_00pctheavy_10meaninterarrivals,single_16,cluster_16_10971159#8b49bae4-b9b3-466d-82f6-b8db25a8e000,234,1,1,False,"[8.0, 23.262456591536385, 6.0, 15.590578156473672, 3.0, 21.122383723998116, 3.0, 18.51808262553164, 3.0, 20.59735201283514, 0.0, 0.0, 0.0, 0.0, 23.123069142836815, 0.0, 0.0, 19.478471038100256, 0.0, 0.0, 0.0, 0.0, 7.943072720829069, 0.0, 0.0, 0.0, 0.0, 5.676753836514856, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]",True
963,1763253072,tpcds_99templates_00pctheavy_10meaninterarrivals,single_16,cluster_16_10971495#8b49bae4-b9b3-466d-82f6-b8db25a8e000,267,1,1,False,"[8.0, 23.262456591536385, 6.0, 15.590416806120011, 3.0, 21.122383723998116, 3.0, 18.51808262553164, 3.0, 20.59735201283514, 0.0, 0.0, 0.0, 0.0, 23.123069142836815, 0.0, 0.0, 19.478471038100256, 0.0, 0.0, 0.0, 0.0, 7.943072720829069, 0.0, 0.0, 0.0, 0.0, 5.676753836514856, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]",True
1138,1763253188,tpcds_99templates_00pctheavy_10meaninterarrivals,single_4,cluster_4_4637750#9654f04d-214c-4356-b976-938bae334a99,90,1,1,False,"[8.0, 23.262456591536385, 6.0, 15.590748969653273, 3.0, 21.122383723998116, 3.0, 18.51808262553164, 3.0, 20.59735201283514, 0.0, 0.0, 0.0, 0.0, 23.123069142836815, 0.0, 0.0, 19.478471038100256, 0.0, 0.0, 0.0, 0.0, 7.943072720829069, 0.0, 0.0, 0.0, 0.0, 5.676753836514856, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]",True
1185,1763253188,tpcds_99templates_00pctheavy_10meaninterarrivals,single_4,cluster_4_4638214#9654f04d-214c-4356-b976-938bae334a99,137,1,1,False,"[8.0, 23.262456591536385, 6.0, 15.590668989336521, 3.0, 21.122383723998116, 3

In [141]:
sys_query_history = pd.read_parquet(
    "/home/markakis/chunkbench/data/runs/1763253188/sys_query_history+cluster_4.parquet"
)
pd.set_option('display.max_columns', None)
sys_query_history[(sys_query_history['query_id'] == 4639738) | (sys_query_history['query_id'] == 4639378)]

,user_id,query_id,query_label,transaction_id,session_id,database_name,query_type,status,result_cache_hit,start_time,end_time,elapsed_time,queue_time,execution_time,error_message,returned_rows,returned_bytes,query_text,redshift_version,usage_limit,compute_type,compile_time,planning_time,lock_wait_time,service_class_id,service_class_name,query_priority,short_query_accelerated,user_query_hash,generic_query_hash,query_hash_version,result_cache_query_id,username,result_offloaded
233,100,4639378,default,515977,1073742455,dev,SELECT,success,False,2025-11-16 01:24:26.959352,2025-11-16 01:30:43.432720,376473368,359835272,16186719,,100,2200,"--1763253188/234\n-- Filename: query001_001.sql\n\nwith customer_total_return as\n(select sr_customer_sk as ctr_customer_sk\n,sr_store_sk as ctr_store_sk\n,sum(SR_REVERSED_CHARGE) as ctr_total_return\nfrom store_returns\n,date_dim\nwhere sr_returned_date_sk = d_date_sk\nand d_year =1998\ngroup by sr_customer_sk\n,sr_store_sk)\n select c_customer_id\nfrom customer_total_return ctr1\n,store\n,customer\nwhere ctr1.ctr_total_return > (select avg(ctr_total_return)*1.2\nfrom customer_total_return ctr2\nwhere ctr1.ctr_store_sk = ctr2.ctr_store_sk)\nand s_store_sk = ctr1.ctr_store_sk\nand s_state = 'GA'\nand ctr1.ctr_customer_sk = c_customer_sk\norder by c_customer_id\nlimit 100;\n\n\n",1.0.136890,,primary,1194,366892438,61,-1,,,,WmF1g/0vepU=,VtgQp+W+DBw=,0,0,admin,f
266,100,4639738,default,517276,1073931376,dev,SELECT,success,False,2025-11-16 01:32:27.107762,2025-11-16 01:33:38.650746,71542984,49902445,9070393,"Query=4639738[child_sequence:4], Query_desc sending CmdAbort",100,2200,"--1763253188/267\n-- Filename: query001_001.sql\n\nwith customer_total_return as\n(select sr_customer_sk as ctr_customer_sk\n,sr_store_sk as ctr_store_sk\n,sum(SR_REVERSED_CHARGE) as ctr_total_return\nfrom store_returns\n,date_dim\nwhere sr_returned_date_sk = d_date_sk\nand d_year =1998\ngroup by sr_customer_sk\n,sr_store_sk)\n select c_customer_id\nfrom customer_total_return ctr1\n,store\n,customer\nwhere ctr1.ctr_total_return > (select avg(ctr_total_return)*1.2\nfrom customer_total_return ctr2\nwhere ctr1.ctr_store_sk = ctr2.ctr_store_sk)\nand s_store_sk = ctr1.ctr_store_sk\nand s_state = 'GA'\nand ctr1.ctr_customer_sk = c_customer_sk\norder by c_customer_id\nlimit 100;\n\n\n",1.0.136890,,primary,1423,57079735,118,-1,,,,WmF1g/0vepU=,VtgQp+W+DBw=,0,0,admin,f


In [142]:
sys_query_explain = pd.read_parquet(
    "/home/markakis/chunkbench/data/runs/1763253188/sys_query_explain+cluster_4.parquet"
)
pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', None)
query_df = sys_query_explain[(sys_query_explain['query_id'] == 4639738) | (sys_query_explain['query_id'] == 4639378)]
query_df

,userid,query_id,child_query_sequence,plan_node_id,plan_parent_id,plan_node,plan_info
1418,100,4639378,1,1,0,XN HashAggregate (cost=3969395.75..4038277.05 rows=27552522 width=24),
1419,100,4639378,1,2,1,-> XN Hash Join DS_DIST_ALL_NONE (cost=918.83..3620058.31 rows=46578325 width=24),"Hash Cond: (""outer"".sr_returned_date_sk = ""inner"".d_date_sk)"
1420,100,4639378,1,3,2,-> XN Seq Scan ext_tpcds1000.store_returns (cost=0.00..2879997.76 rows=275525210 width=28),Filter: (sr_returned_date_sk IS NOT NULL)
1421,100,4639378,1,4,2,-> XN Hash (cost=913.11..913.11 rows=352 width=4),
1422,100,4639378,1,5,4,-> XN Seq Scan ext_tpcds1000.date_dim (cost=0.00..913.11 rows=352 width=4),Filter: (d_year = 1998)
1423,100,4639378,4,1,0,XN Limit (cost=1000890408675.85..1000890408676.10 rows=100 width=20),
1424,100,4639378,4,2,1,-> XN Merge (cost=1000890408675.85..1000890412359.13 rows=1473313 width=20),Merge Key: customer.c_customer_id
1425,100,4639378,4,3,2,-> XN Network (cost=1000890408675.85..1000890412359.13 rows=1473313 width=20),Send to leader
1426,100,4639378,4,4,3,-> XN Sort (cost=1000890408675.85..1000890412359.13 rows=1473313 width=20),Sort Key: customer.c_customer_id
1427,100,4639378,4,5,4,-> XN Hash Join DS_DIST_INNER (cost=6100651.68..890257730.28 rows=1473313 width=20),"Inner Dist Key: volt_tt_643ac1772a37c.sr_customer_sk Hash Cond: (""outer"".c_customer_sk = ""inner"".sr_customer_sk)"


In [61]:
from autoslo.query_plans.parse_plan import plan_summary, parse_one_plan
from typing import Optional

plan_steps = query_df.sort_values("plan_node_id")[
                    "plan_node"
].tolist()
verbose_plan, _, _ = parse_one_plan(plan_steps, analyze=False)
alias_dict: dict[str, Optional[str]] = {}
verbose_plan.parse_lines_recursively(
    schema_name='ext_tpcds1000',
    alias_dict=alias_dict,
)
verbose_plan.parse_columns_bottom_up(
    alias_dict=alias_dict,
)

# Get the tables.
tables, _, _ = plan_summary(verbose_plan)
verbose_plan_dict = verbose_plan.as_serializable()
verbose_plan_dict["tables"] = sorted(list(tables))
verbose_plan_dict["num_tables"] = len(tables)

In [62]:
verbose_plan_dict

{'plain_content': [],
 'plan_parameters': {'op_name': 'XN HashAggregate',
  'est_startup_cost': 12001202759.72,
  'est_cost': 12001226947.71,
  'est_card': 9675197.0,
  'est_width': 25.0,
  'act_children_card': 1.0,
  'est_children_card': 639888503989.0},
 'children': [{'plain_content': [],
   'plan_parameters': {'op_name': 'XN Merge',
    'est_startup_cost': 1010844728766.56,
    'est_cost': 1010844728931.9,
    'est_card': 66137.0,
    'est_width': 128.0,
    'act_children_card': 1.0,
    'est_children_card': 1.0},
   'children': [],
   'plan_runtime': 0},
  {'plain_content': [],
   'plan_parameters': {'op_name': 'XN Hash Join DS_BCAST_INNER',
    'est_startup_cost': 200943.95,
    'est_cost': 12001130195.74,
    'est_card': 9675197.0,
    'est_width': 25.0,
    'act_children_card': 1.0,
    'est_children_card': 4.207936598802e+18},
   'children': [{'plain_content': [],
     'plan_parameters': {'op_name': 'XN Hash Join DS_DIST_ALL_NONE',
      'est_startup_cost': 918.83,
      'est_c